In [13]:
import numpy as np
import matplotlib.pyplot as plt

import h5py
import os
from pathlib import Path

from enum import Enum
import re

from scipy.stats import skew, kurtosis
from scipy.fft import fft, fftfreq

In [14]:
mat_file = r'E:\Thesis\thesis_code\data\rp\100Mbps\ethernet_packets_1_2500.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\ethernet_packets_2501_5000.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\ethernet_packets_5001_7500.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\ethernet_packets_7501_10000.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\ethernet_packets_10001_12500.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\air\ethernet_packets_1250_5cm.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\air\ethernet_packets_1250_10cm.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\air\ethernet_packets_1250_15cm.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\air\ethernet_packets_1250_20cm.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\air\ethernet_packets_1250_25cm.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\air\ethernet_packets_1250_30cm.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\air\ethernet_packets_1250_35cm.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\air\ethernet_packets_1250_40cm.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\air\ethernet_packets_1250_45cm.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\air\ethernet_packets_1250_50cm.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\water\ethernet_packets_1250_5cm.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\water\ethernet_packets_1250_10cm.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\water\ethernet_packets_1250_15cm.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\water\ethernet_packets_1250_20cm.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\water\ethernet_packets_1250_25cm.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\water\ethernet_packets_1250_30cm.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\water\ethernet_packets_1250_35cm.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\water\ethernet_packets_1250_40cm.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\water\ethernet_packets_1250_45cm.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\water\ethernet_packets_1250_50cm.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\tapped\ethernet_packets_2500_0.5m.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\tapped\ethernet_packets_2500_1m.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\tapped\ethernet_packets_2500_1.5m.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\tapped\ethernet_packets_2500_2m.mat'


In [15]:
with h5py.File(mat_file, 'r') as f:
    reference_packet = np.array(f['packets'])  # shape: (recordLength,)
    # Access metadata
    
    metadata = f['metadata']
    sample_rate = metadata['sample_rate'][0][0]
    trigger_level = metadata['trigger_level'][0][0]
    record_length = metadata['record_length'][0][0]
    num_frames = metadata['num_frames'][0][0]

print(f"Sample Rate: {sample_rate} Hz")
print(f"Trigger Level: {trigger_level} V")
print(f"Record Length: {record_length} samples")
print(f"Number of Frames: {num_frames}")

Sample Rate: 125000000.0 Hz
Trigger Level: 1.5 V
Record Length: 16384.0 samples
Number of Frames: 2500.0


In [16]:
for packet in reference_packet:
    len_packet = len(packet)
    print(f"Packet length: {len_packet} samples")

Packet length: 16384 samples
Packet length: 16384 samples
Packet length: 16384 samples
Packet length: 16384 samples
Packet length: 16384 samples
Packet length: 16384 samples
Packet length: 16384 samples
Packet length: 16384 samples
Packet length: 16384 samples
Packet length: 16384 samples
Packet length: 16384 samples
Packet length: 16384 samples
Packet length: 16384 samples
Packet length: 16384 samples
Packet length: 16384 samples
Packet length: 16384 samples
Packet length: 16384 samples
Packet length: 16384 samples
Packet length: 16384 samples
Packet length: 16384 samples
Packet length: 16384 samples
Packet length: 16384 samples
Packet length: 16384 samples
Packet length: 16384 samples
Packet length: 16384 samples
Packet length: 16384 samples
Packet length: 16384 samples
Packet length: 16384 samples
Packet length: 16384 samples
Packet length: 16384 samples
Packet length: 16384 samples
Packet length: 16384 samples
Packet length: 16384 samples
Packet length: 16384 samples
Packet length:

### Exract Signal Region

In [17]:
def extract_signal_region(signal, threshold=0.1):
    """Trims the quiet parts of the signal before and after the packet."""
    active = np.abs(signal) > threshold
    indices = np.where(active)[0]
    if indices.size == 0:
        return np.array([])
    return signal[indices[0] : indices[-1] + 1]

### Ideal Packet Length

In [18]:
sample_rate = 1.25e8  # 1.25GHz
samples_per_bit = int(sample_rate / 10e6)  # 10Mbps
samples_per_bit

12

In [19]:
(sample_rate / 10e6)

12.5

In [20]:
ideal_packet_length = (sample_rate / 10e6)*(8+148+4)*8
ideal_packet_length

16000.0

### Sequence Padding 
- longer packets are discarded


In [23]:
def length_standardization(extracted_signal, target_length=16384):
    """
    Standardizes signal to target_length. 
    Validates that the input is within a +/- 500 sample tolerance before padding.
    """
    current_len = len(extracted_signal)
    tolerance = 500 

    # 1. Check if it's too long
    if current_len > (target_length + tolerance):
        raise ValueError(f"Signal ({current_len}) exceeds target + tolerance.")

    # 2. Check if it's too short
    if current_len < (target_length - tolerance):
        raise ValueError(f"Signal ({current_len}) is below target - tolerance.")

    # 3. Standardize
    if current_len > target_length:
        # If within tolerance but slightly over, clip it
        return extracted_signal[:target_length]
    else:
        # If within tolerance but slightly under, pad it
        pad_size = target_length - current_len
        return np.pad(extracted_signal, (0, pad_size), mode='constant')

### Extract Time domain Features

In [24]:
def extract_time_domain_features(signal):
    """Extract time-domain statistical features"""
    features = {}
    
    # Basic statistical moments
    features['std'] = np.std(signal)
    features['skewness'] = skew(signal)
    features['rms'] = np.sqrt(np.mean(signal**2))
    features['mean'] = np.mean(signal)
    features['kurtosis'] = kurtosis(signal)
    features['peak'] = np.max(np.abs(signal))
    
    # Shape factors
    features['shape_factor'] = features['rms'] / np.mean(np.abs(signal)) if np.mean(np.abs(signal)) != 0 else 0
    features['impulse_factor'] = features['peak'] / np.mean(np.abs(signal)) if np.mean(np.abs(signal)) != 0 else 0
    features['crest_factor'] = features['peak'] / features['rms'] if features['rms'] != 0 else 0
    features['clearance_factor'] = features['peak'] / (np.mean(np.sqrt(np.abs(signal)))**2) if np.mean(np.sqrt(np.abs(signal))) != 0 else 0
    
    return features

### Extract Frequency domain features

In [25]:
def extract_frequency_domain_features(signal, fs):
    """Extract frequency-domain features"""
    # Compute FFT
    fft_signal = fft(signal)
    freqs = fftfreq(len(signal), 1/fs)
    
    # Get positive frequencies only
    positive_freq_idx = freqs > 0
    fft_positive = fft_signal[positive_freq_idx]
    freqs_positive = freqs[positive_freq_idx]
    
    # Find top 10 harmonic components
    magnitudes = np.abs(fft_positive)
    top_indices = np.argsort(magnitudes)[-10:]
    
    features = {}
    for i, idx in enumerate(top_indices):
        features[f'freq_mag_{i}'] = magnitudes[idx]
        features[f'freq_val_{i}'] = freqs_positive[idx]
    
    return features

### Label Extracton from File Name

In [26]:
class ChannelCondition(Enum):
    NORMAL = 0
    AIR = 1
    WATER = 2
    TAPPED = 3

In [27]:
def extract_label_from_filename(filepath):
    filename = filepath.lower()

    # TYPE LABEL
    if 'air' in filename:
        anomaly_type = ChannelCondition.AIR.value
    elif 'water' in filename:
        anomaly_type = ChannelCondition.WATER.value
    elif 'tapped' in filename:
        anomaly_type = ChannelCondition.TAPPED.value
    else:
        anomaly_type = ChannelCondition.NORMAL.value  # normal

    # LENGTH / DISTANCE
    
    match = re.search(r'_(\d+(?:\.\d+)?)(cm|m)\.mat$', filename)

    if match:
        value = float(match.group(1))
        unit = match.group(2)

        if unit == 'm':
            value *= 100  # convert to cm

        anomaly_length = value
    else:
        anomaly_length = 0

    return anomaly_type, anomaly_length

In [28]:
# extract_label_from_filename(mat_file)

In [30]:
from pathlib import Path

root_dir = Path(r"E:\Thesis\thesis_code\data\rp\100Mbps")

for mat_file in root_dir.rglob("*.mat"):
    print(mat_file)
    print(extract_label_from_filename(str(mat_file)))

E:\Thesis\thesis_code\data\rp\100Mbps\ethernet_packets_1_2500.mat
(0, 0)
E:\Thesis\thesis_code\data\rp\100Mbps\ethernet_packets_2501_5000.mat
(0, 0)
E:\Thesis\thesis_code\data\rp\100Mbps\ethernet_packets_5001_7500.mat
(0, 0)
E:\Thesis\thesis_code\data\rp\100Mbps\ethernet_packets_7501_10000.mat
(0, 0)
E:\Thesis\thesis_code\data\rp\100Mbps\air\ethernet_packets_1000_10cm.mat
(1, 10.0)
E:\Thesis\thesis_code\data\rp\100Mbps\air\ethernet_packets_1000_15cm.mat
(1, 15.0)
E:\Thesis\thesis_code\data\rp\100Mbps\air\ethernet_packets_1000_20cm.mat
(1, 20.0)
E:\Thesis\thesis_code\data\rp\100Mbps\air\ethernet_packets_1000_25cm.mat
(1, 25.0)
E:\Thesis\thesis_code\data\rp\100Mbps\air\ethernet_packets_1000_30cm.mat
(1, 30.0)
E:\Thesis\thesis_code\data\rp\100Mbps\air\ethernet_packets_1000_35cm.mat
(1, 35.0)
E:\Thesis\thesis_code\data\rp\100Mbps\air\ethernet_packets_1000_40cm.mat
(1, 40.0)
E:\Thesis\thesis_code\data\rp\100Mbps\air\ethernet_packets_1000_45cm.mat
(1, 45.0)
E:\Thesis\thesis_code\data\rp\100M

In [34]:
output_file = r'E:\Thesis\thesis_code\data\rp\100Mbps_features.h5'

In [32]:

def extract_features_from_mat_file(root_dir, output_file):
    
    with h5py.File(output_file, 'x') as out_f:

        feature_dim = 10 + 20  # 10 time + (10 mag + 10 freq)

        features_dset = out_f.create_dataset(
            'features',
            shape=(0, feature_dim),
            maxshape=(None, feature_dim),
            dtype='float32',
            chunks=True
        )

        labels_type = out_f.create_dataset(
            'label_type',
            shape=(0,),
            maxshape=(None,),
            dtype='int32',
            chunks=True
        )

        labels_length = out_f.create_dataset(
            'label_length',
            shape=(0,),
            maxshape=(None,),
            dtype='float32',
            chunks=True
        )

        current_size = 0

        for mat_file in root_dir.rglob("*.mat"):
            print(f"\nProcessing: {mat_file}")

            anomaly_type, anomaly_length = extract_label_from_filename(str(mat_file))

            with h5py.File(mat_file, 'r') as f:
                packets = f['packets']
                metadata = f['metadata']

                fs = metadata['sample_rate'][0][0]
                num_frames = packets.shape[0]

                batch_size = 100
                for i in range(0, num_frames, batch_size):
                    

                    try:
                        batch = packets[i:i+batch_size]  # lazy load
                        features_batch = []
                        types_batch = []
                        lengths_batch = []
                        for j, signal in enumerate(batch):
                            try:
                                # --- preprocessing ---
                                signal = extract_signal_region(signal)


                                signal = length_standardization(signal)

                                # --- feature extraction ---
                                time_feat = extract_time_domain_features(signal)
                                freq_feat = extract_frequency_domain_features(signal, fs)

                                # Combine features
                                combined = list(time_feat.values()) + list(freq_feat.values())
                                combined = np.array(combined, dtype=np.float32)

                                # --- store ---
                                # features_dset.resize(current_size + 1, axis=0)
                                # labels_type.resize(current_size + 1, axis=0)
                                # labels_length.resize(current_size + 1, axis=0)

                                # features_dset[current_size] = combined
                                # labels_type[current_size] = anomaly_type
                                # labels_length[current_size] = anomaly_length

                                features_batch.append(combined)
                                types_batch.append(anomaly_type)
                                lengths_batch.append(anomaly_length)
                            except Exception as e:
                                print(f"Skipping signal in frame {i+j}: {e}")
                                continue

                        # --- ONLY write valid samples ---
                        valid_count = len(features_batch)


                        if valid_count == 0:
                            continue

                        new_size = current_size + valid_count

                        features_dset.resize(new_size, axis=0)
                        labels_type.resize(new_size, axis=0)
                        labels_length.resize(new_size, axis=0)

                        features_dset[current_size:new_size] = features_batch
                        labels_type[current_size:new_size] = types_batch
                        labels_length[current_size:new_size] = lengths_batch

                        current_size = new_size

                    except Exception as e:
                        print(f"Skipping frame {i}: {e}")
                        continue

                    if i % 100 == 0:
                        print(f"Processed {i}/{num_frames} | Added {valid_count} samples")
                    # break
            # break
    print("\n✅ Feature extraction completed!")

In [ ]:
feature_labels = [
    # --- Time-domain features (0–9) ---
    "std",                 # 0
    "skewness",            # 1
    "rms",                 # 2
    "mean",                # 3
    "kurtosis",            # 4
    "peak",                # 5
    "shape_factor",        # 6
    "impulse_factor",      # 7
    "crest_factor",        # 8
    "clearance_factor",    # 9

    # --- Frequency-domain magnitude features (10–19) ---
    "freq_mag_0",          # 10
    "freq_mag_1",          # 11
    "freq_mag_2",          # 12
    "freq_mag_3",          # 13
    "freq_mag_4",          # 14
    "freq_mag_5",          # 15
    "freq_mag_6",          # 16
    "freq_mag_7",          # 17
    "freq_mag_8",          # 18
    "freq_mag_9",          # 19

    # --- Frequency-domain frequency values (20–29) ---
    "freq_val_0",          # 20
    "freq_val_1",          # 21
    "freq_val_2",          # 22
    "freq_val_3",          # 23
    "freq_val_4",          # 24
    "freq_val_5",          # 25
    "freq_val_6",          # 26
    "freq_val_7",          # 27
    "freq_val_8",          # 28
    "freq_val_9"           # 29
]

In [33]:
# import cProfile
# cProfile.run('extract_features_from_mat_file(root_dir, output_file)')

In [35]:
extract_features_from_mat_file(root_dir, output_file)


Processing: E:\Thesis\thesis_code\data\rp\100Mbps\ethernet_packets_1_2500.mat
Processed 0/2500 | Added 100 samples
Processed 100/2500 | Added 100 samples
Processed 200/2500 | Added 100 samples
Processed 300/2500 | Added 100 samples
Processed 400/2500 | Added 100 samples
Processed 500/2500 | Added 100 samples
Processed 600/2500 | Added 100 samples
Processed 700/2500 | Added 100 samples
Processed 800/2500 | Added 100 samples
Processed 900/2500 | Added 100 samples
Processed 1000/2500 | Added 100 samples
Processed 1100/2500 | Added 100 samples
Processed 1200/2500 | Added 100 samples
Processed 1300/2500 | Added 100 samples
Processed 1400/2500 | Added 100 samples
Processed 1500/2500 | Added 100 samples
Processed 1600/2500 | Added 100 samples
Processed 1700/2500 | Added 100 samples
Processed 1800/2500 | Added 100 samples
Processed 1900/2500 | Added 100 samples
Processed 2000/2500 | Added 100 samples
Processed 2100/2500 | Added 100 samples
Processed 2200/2500 | Added 100 samples
Processed 230